In [1]:
import sympy as sp

m,n = sp.symbols('m, n', real=True)
tau1 = sp.Function('tau_1', real=True)(m, n)
tau2 = sp.Function('tau_2', real=True, positive=True)(m, n)
tau = tau1 + sp.I*tau2
H3 = sp.Function('H3', real=True)(m, n)
F3 = sp.Function('F3', real=True)(m, n)
G3  = tau2**sp.Rational(-1,2) * (F3 - (tau1 + sp.I*tau2)*H3)
G3b = tau2**sp.Rational(-1,2) * (F3 - (tau1 - sp.I*tau2)*H3)


# Define the substitutions for the functions and their derivatives
H3_sym = sp.Symbol('H_{3}')
F3_sym = sp.Symbol('F_{3}')
tau1_sym = sp.Symbol(r'\tau_1')
tau2_sym = sp.Symbol(r'\tau_2')

dH3_m = sp.Symbol(r'\nabla_{m} H_3')
dH3_n = sp.Symbol(r'\nabla_{n} H_3')
dF3_m = sp.Symbol(r'\nabla_{m} F_3')
dF3_n = sp.Symbol(r'\nabla_{n} F_3')
dtau1_m = sp.Symbol(r'\nabla_{m} \tau_1')
dtau1_n = sp.Symbol(r'\nabla_{n} \tau_1')
dtau2_m = sp.Symbol(r'\nabla_{m} \tau_2')
dtau2_n = sp.Symbol(r'\nabla_{n} \tau_2')

function_subs = [
    (H3, H3_sym),
    (F3, F3_sym),
    (tau1, tau1_sym),
    (tau2, tau2_sym),
]
deriv_subs = [
    (sp.diff(H3, m), dH3_m),
    (sp.diff(H3, n), dH3_n),
    (sp.diff(F3, m), dF3_m),
    (sp.diff(F3, n), dF3_n),
    (sp.diff(tau1, m), dtau1_m),
    (sp.diff(tau1, n), dtau1_n),
    (sp.diff(tau2, m), dtau2_m),
    (sp.diff(tau2, n), dtau2_n),
]


In [2]:
G3

(-(tau_1(m, n) + I*tau_2(m, n))*H3(m, n) + F3(m, n))/sqrt(tau_2(m, n))

In [3]:
# |D G3|^2 = D G3 * D G3b
Q_m = - sp.Rational(1, 2) * sp.diff(tau1, m) / tau2
Q_n = - sp.Rational(1, 2) * sp.diff(tau1, n) / tau2

DG_3 = sp.diff(G3, n) - sp.I * Q_n * G3
DGb_3 = sp.diff(G3b, m) + sp.I * Q_m * G3b

prod = DGb_3 * DG_3
prod = sp.simplify(prod)

# remove imagainary components because they don't contribute
prod = sp.re(sp.expand(prod))
prod = prod.replace(lambda e: e.func == sp.re, lambda e: e.args[0])
prod = prod.replace(lambda e: e.func == sp.im, lambda e: sp.S.Zero)

In [4]:
prod

F3(m, n)**2*Derivative(tau_1(m, n), m)*Derivative(tau_1(m, n), n)/(4*tau_2(m, n)**3) + F3(m, n)**2*Derivative(tau_2(m, n), m)*Derivative(tau_2(m, n), n)/(4*tau_2(m, n)**3) - F3(m, n)*H3(m, n)*tau_1(m, n)*Derivative(tau_1(m, n), m)*Derivative(tau_1(m, n), n)/(2*tau_2(m, n)**3) - F3(m, n)*H3(m, n)*tau_1(m, n)*Derivative(tau_2(m, n), m)*Derivative(tau_2(m, n), n)/(2*tau_2(m, n)**3) + F3(m, n)*tau_1(m, n)*Derivative(H3(m, n), m)*Derivative(tau_2(m, n), n)/(2*tau_2(m, n)**2) + F3(m, n)*tau_1(m, n)*Derivative(H3(m, n), n)*Derivative(tau_2(m, n), m)/(2*tau_2(m, n)**2) - F3(m, n)*Derivative(H3(m, n), m)*Derivative(tau_1(m, n), n)/(2*tau_2(m, n)) - F3(m, n)*Derivative(H3(m, n), n)*Derivative(tau_1(m, n), m)/(2*tau_2(m, n)) - F3(m, n)*Derivative(F3(m, n), m)*Derivative(tau_2(m, n), n)/(2*tau_2(m, n)**2) - F3(m, n)*Derivative(F3(m, n), n)*Derivative(tau_2(m, n), m)/(2*tau_2(m, n)**2) + H3(m, n)**2*tau_1(m, n)**2*Derivative(tau_1(m, n), m)*Derivative(tau_1(m, n), n)/(4*tau_2(m, n)**3) + H3(m, n)**

In [5]:
# Metric
M = sp.Matrix([[1/tau2, tau1/tau2], [tau1/tau2, (tau1**2 + tau2**2)/tau2]])
M_inv = M.inv()

# Here we are defining Gamma^a_{m/n b}, in order (a,b)
Gamma_m = sp.Rational(1,2) * sp.simplify(M_inv * sp.diff(M,m))
Gamma_n = sp.Rational(1,2) * sp.simplify(M_inv * sp.diff(M,n))

In [6]:
nabla_m_F4_down_1 = sp.diff(H3, m) - Gamma_m[0,0]*H3 - Gamma_m[1,0]* F3
nabla_n_F4_down_1 = sp.diff(H3, n) - Gamma_n[0,0]*H3 - Gamma_n[1,0]* F3
nabla_m_F4_down_2 = sp.diff(F3, m) - Gamma_m[0,1]*H3 - Gamma_m[1,1]* F3
nabla_n_F4_down_2 = sp.diff(F3, n) - Gamma_n[0,1]*H3 - Gamma_n[1,1]* F3

nabla_m_F4_up_1 = M_inv[0,0]*nabla_m_F4_down_1 + M_inv[0,1]*nabla_m_F4_down_2
nabla_n_F4_up_1 = M_inv[0,0]*nabla_n_F4_down_1 + M_inv[0,1]*nabla_n_F4_down_2
nabla_m_F4_up_2 = M_inv[1,0]*nabla_m_F4_down_1 + M_inv[1,1]*nabla_m_F4_down_2
nabla_n_F4_up_2 = M_inv[1,0]*nabla_n_F4_down_1 + M_inv[1,1]*nabla_n_F4_down_2

nabla_m_F4_down_1_sym = sp.Symbol(r'\nabla_{m} (F_4)_{1}')
nabla_n_F4_down_1_sym = sp.Symbol(r'\nabla_{n} (F_4)_{1}')
nabla_m_F4_down_2_sym = sp.Symbol(r'\nabla_{m} (F_4)_{2}')
nabla_n_F4_down_2_sym = sp.Symbol(r'\nabla_{n} (F_4)_{2}')
nabla_m_F4_up_1_sym = sp.Symbol(r'\nabla_{m} (F_4)^{1}')
nabla_n_F4_up_1_sym = sp.Symbol(r'\nabla_{n} (F_4)^{1}')
nabla_m_F4_up_2_sym = sp.Symbol(r'\nabla_{m} (F_4)^{2}')
nabla_n_F4_up_2_sym = sp.Symbol(r'\nabla_{n} (F_4)^{2}')

In [7]:
comp = M_inv[0,0]*nabla_m_F4_down_1 * nabla_n_F4_down_1 + M_inv[0,1]*nabla_m_F4_down_1 * nabla_n_F4_down_2 + M_inv[1,0]*nabla_m_F4_down_2 * nabla_n_F4_down_1 + M_inv[1,1]*nabla_m_F4_down_2 * nabla_n_F4_down_2
comp = sp.expand(sp.simplify(comp))

In [8]:
# nabla_m_F4_1_sym = sp.Symbol(r'\nabla_{m} (F_4)_{1}')
# nabla_m_F4_1 = dH3_m - Gamma_m[0,0]*H3_sym - Gamma_m[1,0]* F3_sym
# nabla_m_F4_1 = nabla_m_F4_1.subs(deriv_subs).subs(function_subs)
# nabla_m_F4_1_subs = {dH3_m: nabla_m_F4_1_sym-(nabla_m_F4_1 - dH3_m)}

# nabla_n_F4_1_sym = sp.Symbol(r'\nabla_{n} (F_4)_{1}')
# nabla_n_F4_1 = dH3_n - Gamma_n[0,0]*H3_sym - Gamma_n[1,0]* F3_sym
# nabla_n_F4_1 = nabla_n_F4_1.subs(deriv_subs).subs(function_subs)
# nabla_n_F4_1_subs = {dH3_n: nabla_n_F4_1_sym-(nabla_n_F4_1 - dH3_n)}

# nabla_m_F4_2_sym = sp.Symbol(r'\nabla_{m} (F_4)_{2}')
# nabla_m_F4_2 = dF3_m - Gamma_m[0,1]*H3_sym - Gamma_m[1,1]* F3_sym
# nabla_m_F4_2 = nabla_m_F4_2.subs(deriv_subs).subs(function_subs)
# nabla_m_F4_2_subs = {dF3_m: nabla_m_F4_2_sym-(nabla_m_F4_2 - dF3_m)}

# nabla_n_F4_2_sym = sp.Symbol(r'\nabla_{n} (F_4)_{2}')
# nabla_n_F4_2 = dF3_n - Gamma_n[0,1]*H3_sym - Gamma_n[1,1]* F3_sym
# nabla_n_F4_2 = nabla_n_F4_2.subs(deriv_subs).subs(function_subs)
# nabla_n_F4_2_subs = {dF3_n: nabla_n_F4_2_sym-(nabla_n_F4_2 - dF3_n)}

# Replace functions with symbols

In [9]:
prod

F3(m, n)**2*Derivative(tau_1(m, n), m)*Derivative(tau_1(m, n), n)/(4*tau_2(m, n)**3) + F3(m, n)**2*Derivative(tau_2(m, n), m)*Derivative(tau_2(m, n), n)/(4*tau_2(m, n)**3) - F3(m, n)*H3(m, n)*tau_1(m, n)*Derivative(tau_1(m, n), m)*Derivative(tau_1(m, n), n)/(2*tau_2(m, n)**3) - F3(m, n)*H3(m, n)*tau_1(m, n)*Derivative(tau_2(m, n), m)*Derivative(tau_2(m, n), n)/(2*tau_2(m, n)**3) + F3(m, n)*tau_1(m, n)*Derivative(H3(m, n), m)*Derivative(tau_2(m, n), n)/(2*tau_2(m, n)**2) + F3(m, n)*tau_1(m, n)*Derivative(H3(m, n), n)*Derivative(tau_2(m, n), m)/(2*tau_2(m, n)**2) - F3(m, n)*Derivative(H3(m, n), m)*Derivative(tau_1(m, n), n)/(2*tau_2(m, n)) - F3(m, n)*Derivative(H3(m, n), n)*Derivative(tau_1(m, n), m)/(2*tau_2(m, n)) - F3(m, n)*Derivative(F3(m, n), m)*Derivative(tau_2(m, n), n)/(2*tau_2(m, n)**2) - F3(m, n)*Derivative(F3(m, n), n)*Derivative(tau_2(m, n), m)/(2*tau_2(m, n)**2) + H3(m, n)**2*tau_1(m, n)**2*Derivative(tau_1(m, n), m)*Derivative(tau_1(m, n), n)/(4*tau_2(m, n)**3) + H3(m, n)**

In [10]:
comp

F3(m, n)**2*Derivative(tau_1(m, n), m)*Derivative(tau_1(m, n), n)/(4*tau_2(m, n)**3) + F3(m, n)**2*Derivative(tau_2(m, n), m)*Derivative(tau_2(m, n), n)/(4*tau_2(m, n)**3) - F3(m, n)*H3(m, n)*tau_1(m, n)*Derivative(tau_1(m, n), m)*Derivative(tau_1(m, n), n)/(2*tau_2(m, n)**3) - F3(m, n)*H3(m, n)*tau_1(m, n)*Derivative(tau_2(m, n), m)*Derivative(tau_2(m, n), n)/(2*tau_2(m, n)**3) + F3(m, n)*tau_1(m, n)*Derivative(H3(m, n), m)*Derivative(tau_2(m, n), n)/(2*tau_2(m, n)**2) + F3(m, n)*tau_1(m, n)*Derivative(H3(m, n), n)*Derivative(tau_2(m, n), m)/(2*tau_2(m, n)**2) - F3(m, n)*Derivative(H3(m, n), m)*Derivative(tau_1(m, n), n)/(2*tau_2(m, n)) - F3(m, n)*Derivative(H3(m, n), n)*Derivative(tau_1(m, n), m)/(2*tau_2(m, n)) - F3(m, n)*Derivative(F3(m, n), m)*Derivative(tau_2(m, n), n)/(2*tau_2(m, n)**2) - F3(m, n)*Derivative(F3(m, n), n)*Derivative(tau_2(m, n), m)/(2*tau_2(m, n)**2) + H3(m, n)**2*tau_1(m, n)**2*Derivative(tau_1(m, n), m)*Derivative(tau_1(m, n), n)/(4*tau_2(m, n)**3) + H3(m, n)**

In [11]:
res = sp.expand(sp.simplify(comp - prod))
res_sym = sp.expand(sp.simplify(res.subs(deriv_subs).subs(function_subs)))
res_sym

0

# Covariantize derivatives

In [12]:
covariantization_subs = {dH3_m: nabla_m_F4_down_1_sym -  (nabla_m_F4_down_1.subs(deriv_subs).subs(function_subs) - dH3_m),
 dH3_n: nabla_n_F4_down_1_sym -  (nabla_n_F4_down_1.subs(deriv_subs).subs(function_subs) - dH3_n),
 dF3_m: nabla_m_F4_down_2_sym -  (nabla_m_F4_down_2.subs(deriv_subs).subs(function_subs) - dF3_m),
 dF3_n: nabla_n_F4_down_2_sym -  (nabla_n_F4_down_2.subs(deriv_subs).subs(function_subs) - dF3_n),}

In [13]:
res_sym = sp.expand(sp.simplify(res_sym.subs(covariantization_subs)))

In [14]:
nabla_lower_to_upper = [
    (nabla_m_F4_down_1_sym, 1/tau2_sym * nabla_m_F4_up_1_sym + tau1_sym/tau2_sym * nabla_m_F4_up_2_sym),
    (nabla_m_F4_down_2_sym, tau1_sym/tau2_sym * nabla_m_F4_up_1_sym + (tau1_sym**2 + tau2_sym**2)/tau2_sym * nabla_m_F4_up_2_sym),
    (nabla_n_F4_down_1_sym, 1/tau2_sym * nabla_n_F4_up_1_sym + tau1_sym/tau2_sym * nabla_n_F4_up_2_sym),
    (nabla_n_F4_down_2_sym, tau1_sym/tau2_sym * nabla_n_F4_up_1_sym + (tau1_sym**2 + tau2_sym**2)/tau2_sym * nabla_n_F4_up_2_sym),
]

In [15]:
res_sym = sp.expand(sp.simplify(res_sym.subs(nabla_lower_to_upper)))

In [16]:
F4up_1 = sp.Symbol(r'(F_{4}^{1})')
F4up_2 = sp.Symbol(r'(F_{4}^{2})')

In [17]:
HF_2_F4= [
    (H3_sym, 1/tau2_sym * F4up_1 + tau1_sym/tau2_sym * F4up_2),
    (F3_sym, tau1_sym/tau2_sym * F4up_1 + (tau1_sym**2 + tau2_sym**2)/tau2_sym * F4up_2),
]

In [18]:
res_sym = sp.expand(sp.simplify(res_sym.subs(HF_2_F4)))

In [19]:
res_sym

0

In [20]:
sp.collect(sp.expand(res_sym), [F4up_1, F4up_2])

0

We have shown that $|D G_3|^2 = (\nabla F_4)_a(\nabla F_4)_b g^{ab} $, so that the overall contribution is

$$
6R^2(t_8t_8-\tfrac14 \epsilon_8\epsilon_8) (\nabla F_4)_a(\nabla F_4)_b g^{ab}
$$